# ConfigManager: Centralized Runtime Configuration

This notebook demonstrates the `ConfigManager` API (ADR-034), the single
authoritative source for all runtime configuration in `calibrated_explanations`.

**Topics covered**

1. Snapshot semantics and basic API
2. Deterministic precedence: override → env → pyproject → default
3. Supported environment variables and their pyproject.toml equivalents
4. Diagnostic export with `export_effective()`
5. Plugin-specific configuration via `CE_PLUGIN_CONFIG_JSON`
6. Strict vs. permissive validation
7. Process-level singleton lifecycle
8. Telemetry diagnostic mode

> **Note:** `ConfigManager` is an infrastructure primitive. Its stable import
> path is `calibrated_explanations.core.config_manager`. It is *not* promoted
> to the root namespace by design (ADR-034 §7).


## Setup

In [1]:
import json
import os
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

from calibrated_explanations.core.config_manager import (
    ConfigManager,
    get_process_config_manager,
    init_process_config_manager,
    reset_process_config_manager_for_testing,
)
from calibrated_explanations import __version__

print(f"calibrated_explanations {__version__}")
print("ConfigManager imported successfully")


calibrated_explanations 0.11.3
ConfigManager imported successfully


## 1. Snapshot semantics

`ConfigManager` captures environment variables and `pyproject.toml` content
**once** at construction time via `ConfigManager.from_sources()`. Later
changes to `os.environ` are **not** observed by an already-constructed manager.

This makes configuration deterministic: every lookup returns the same value
it would have returned at process startup.


In [2]:
# Set env var BEFORE creating the manager — it will be captured in the snapshot
os.environ["CE_PLOT_STYLE"] = "plot_spec.default"

cfg = ConfigManager.from_sources()

print("After construction, env() returns the captured snapshot value:")
print(f"  CE_PLOT_STYLE -> {cfg.env('CE_PLOT_STYLE')!r}")

# Change the env var AFTER construction — the manager does NOT observe this
os.environ["CE_PLOT_STYLE"] = "legacy"

print("After mutating os.environ, the manager still returns the snapshot:")
print(f"  manager.env() -> {cfg.env('CE_PLOT_STYLE')!r}   (unchanged)")
print(f"  os.environ    -> {os.environ.get('CE_PLOT_STYLE')!r}  (live)")

os.environ.pop("CE_PLOT_STYLE", None)


After construction, env() returns the captured snapshot value:
  CE_PLOT_STYLE -> 'plot_spec.default'
After mutating os.environ, the manager still returns the snapshot:
  manager.env() -> 'plot_spec.default'   (unchanged)
  os.environ    -> 'legacy'  (live)


'legacy'

In [3]:
# To observe updated env values: reconstruct a new manager
os.environ["CE_INTERVAL_PLUGIN"] = "core.interval.legacy"
cfg2 = ConfigManager.from_sources()

print("Querying env() on a freshly constructed manager:")
print(f"  CE_INTERVAL_PLUGIN -> {cfg2.env('CE_INTERVAL_PLUGIN')!r}")

os.environ.pop("CE_INTERVAL_PLUGIN", None)


Querying env() on a freshly constructed manager:
  CE_INTERVAL_PLUGIN -> 'core.interval.legacy'


'core.interval.legacy'

## 2. Deterministic precedence

Resolution order (highest to lowest):

| Priority | Source |
|----------|--------|
| 1 — highest | Call-site override |
| 2 | Environment variable |
| 3 | `pyproject.toml` (`[tool.calibrated_explanations.*]`) |
| 4 — lowest | Versioned default profile |

`_resolve_effective_key()` drives this logic internally and is used by
`export_effective()`. The `env()` helper exposes just the env-snapshot layer.


In [4]:
# Layer 2: env var set → returned from snapshot
os.environ["CE_PLOT_STYLE"] = "plot_spec.default"
cfg_env = ConfigManager.from_sources()
val, src = cfg_env._resolve_effective_key("CE_PLOT_STYLE")
print(f"env set:     CE_PLOT_STYLE = {val!r}  [{src}]")

# Layer 4: env absent, no pyproject → default profile
os.environ.pop("CE_PLOT_STYLE", None)
cfg_no_env = ConfigManager.from_sources()
val, src = cfg_no_env._resolve_effective_key("CE_PLOT_STYLE")
print(f"env absent:  CE_PLOT_STYLE = {val!r}  [{src}]")

# Layer 1: call-site override beats env var
os.environ["CE_PLOT_STYLE"] = "plot_spec.default"
cfg_ov = ConfigManager.from_sources()
val, src = cfg_ov._resolve_effective_key(
    "CE_PLOT_STYLE", override={"CE_PLOT_STYLE": "my.custom.style"}
)
print(f"override:    CE_PLOT_STYLE = {val!r}  [{src}]")

# Layer 3: pyproject beats default (simulated via direct constructor)
cfg_py = ConfigManager(
    env_snapshot={},
    pyproject_snapshot={"plots": {"style": "my.pyproject.style"}},
)
val, src = cfg_py._resolve_effective_key("CE_PLOT_STYLE")
print(f"pyproject:   CE_PLOT_STYLE = {val!r}  [{src}]")

os.environ.pop("CE_PLOT_STYLE", None)


env set:     CE_PLOT_STYLE = 'plot_spec.default'  [env]
env absent:  CE_PLOT_STYLE = None  [default_profile]
override:    CE_PLOT_STYLE = 'my.custom.style'  [override]
pyproject:   CE_PLOT_STYLE = 'my.pyproject.style'  [pyproject]


'plot_spec.default'

## 3. Supported environment variables

The canonical list of recognized `CE_*` keys lives in `ConfigManager._spec`.
Unknown `CE_*` keys emit a `UserWarning` at construction time and are ignored.


In [5]:
# All keys recognized by the default spec
known = sorted(ConfigManager._spec.known_env_keys)
print(f"Total recognized keys: {len(known)}")
print()
print(f"  {'Key':<45} pyproject.toml path")
print("-" * 80)
for key in known:
    section, py_key, default = ConfigManager._spec.resolution_spec.get(
        key, (None, None, None)
    )
    if section and py_key:
        pyproject_path = f"[{section}].{py_key}"
    else:
        pyproject_path = "(env-only in v0.11.x)"
    print(f"  {key:<45} {pyproject_path}")


Total recognized keys: 26

  Key                                           pyproject.toml path
--------------------------------------------------------------------------------
  CE_CACHE                                      (env-only in v0.11.x)
  CE_DEBUG_TRUST_INVARIANTS                     (env-only in v0.11.x)
  CE_DENY_PLUGIN                                (env-only in v0.11.x)
  CE_EXPLANATION_PLUGIN                         [explanations].factual
  CE_EXPLANATION_PLUGIN_ALTERNATIVE             [explanations].alternative
  CE_EXPLANATION_PLUGIN_ALTERNATIVE_FALLBACKS   [explanations].alternative_fallbacks
  CE_EXPLANATION_PLUGIN_FACTUAL                 [explanations].factual
  CE_EXPLANATION_PLUGIN_FACTUAL_FALLBACKS       [explanations].factual_fallbacks
  CE_EXPLANATION_PLUGIN_FAST                    [explanations].fast
  CE_EXPLANATION_PLUGIN_FAST_FALLBACKS          [explanations].fast_fallbacks
  CE_FEATURE_FILTER                             (env-only in v0.11.x)
  CE_INTERVAL_P

In [6]:
# Unknown CE_* key triggers a UserWarning (not an error)
os.environ["CE_UNKNOWN_FUTURE_KEY"] = "some_value"

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    cfg_unknown = ConfigManager.from_sources()

for w in caught:
    if "CE_UNKNOWN_FUTURE_KEY" in str(w.message):
        print(f"UserWarning: {w.message}")

os.environ.pop("CE_UNKNOWN_FUTURE_KEY", None)


'some_value'

## 4. Diagnostic export with `export_effective()`

`export_effective()` returns a frozen `ResolvedConfigSnapshot` with:

- `values` — every resolved value (secret-like keys auto-redacted)
- `sources` — per-key source attribution (`"env"`, `"pyproject"`, `"default_profile"`)
- `profile_id` and `schema_version` — versioning markers

Use this as the primary diagnostic tool for support bundles and deployment audits.


In [7]:
# Set up a realistic env snapshot
os.environ["CE_PLOT_STYLE"] = "plot_spec.default"
os.environ["CE_PLOT_STYLE_FALLBACKS"] = "legacy"
os.environ["CE_INTERVAL_PLUGIN"] = "core.interval.legacy"
os.environ["CE_TELEMETRY_DIAGNOSTIC_MODE"] = "true"

cfg_export = ConfigManager.from_sources()
snapshot = cfg_export.export_effective()

print(f"Profile:        {snapshot.profile_id}")
print(f"Schema version: {snapshot.schema_version}")
print()

# Show only effective.* resolved values
print(f"{'Key':<45} {'Value':<35} Source")
print("-" * 95)
for key, value in snapshot.values.items():
    if not key.startswith("effective."):
        continue
    short_key = key.replace("effective.", "")
    source = snapshot.sources[key]
    print(f"  {short_key:<43} {str(value):<35} {source}")

os.environ.pop("CE_PLOT_STYLE", None)
os.environ.pop("CE_PLOT_STYLE_FALLBACKS", None)
os.environ.pop("CE_INTERVAL_PLUGIN", None)
os.environ.pop("CE_TELEMETRY_DIAGNOSTIC_MODE", None)


Profile:        default
Schema version: 1

Key                                           Value                               Source
-----------------------------------------------------------------------------------------------
  CE_EXPLANATION_PLUGIN                       None                                default_profile
  CE_EXPLANATION_PLUGIN_FACTUAL               None                                default_profile
  CE_EXPLANATION_PLUGIN_ALTERNATIVE           None                                default_profile
  CE_EXPLANATION_PLUGIN_FAST                  None                                default_profile
  CE_EXPLANATION_PLUGIN_FACTUAL_FALLBACKS     ()                                  default_profile
  CE_EXPLANATION_PLUGIN_ALTERNATIVE_FALLBACKS ()                                  default_profile
  CE_EXPLANATION_PLUGIN_FAST_FALLBACKS        ()                                  default_profile
  CE_INTERVAL_PLUGIN                          core.interval.legacy                env


'true'

In [8]:
# Secret-like keys are automatically redacted in the export
plugin_cfg_blob = json.dumps({
    "my_org.plugin": {"endpoint": "https://example.com", "api_key": "s3cr3t"},
})
os.environ["CE_PLUGIN_CONFIG_JSON"] = plugin_cfg_blob

cfg_redact = ConfigManager.from_sources()
snap_redact = cfg_redact.export_effective()

plugin_export = snap_redact.values.get("effective.plugin_config.my_org.plugin", {})
print("Exported plugin config (api_key is redacted):")
for k, v in dict(plugin_export).items():
    print(f"  {k}: {v!r}")

os.environ.pop("CE_PLUGIN_CONFIG_JSON", None)


Exported plugin config (api_key is redacted):
  endpoint: 'https://example.com'
  api_key: '<redacted>'


'{"my_org.plugin": {"endpoint": "https://example.com", "api_key": "s3cr3t"}}'

## 5. Plugin-specific configuration

Each plugin can have its own key/value config supplied via:

- `CE_PLUGIN_CONFIG_JSON` — JSON blob in the environment (env overrides pyproject)
- `[tool.calibrated_explanations.plugin_configs."<plugin_id>"]` in `pyproject.toml`

The resolved config is deeply immutable (`MappingProxyType`) so trusted
plugins observe a deterministic snapshot and cannot mutate shared state.


In [9]:
# Multiple plugins in one CE_PLUGIN_CONFIG_JSON blob
plugin_blob = json.dumps({
    "demo.factual.v2": {"threshold": 0.05, "max_features": 10},
    "demo.interval.custom": {"confidence": 0.9, "method": "venn_abers"},
})
os.environ["CE_PLUGIN_CONFIG_JSON"] = plugin_blob

cfg_plugin = ConfigManager.from_sources()

print("Configured plugin IDs:", cfg_plugin.configured_plugin_ids())
print()

for pid in cfg_plugin.configured_plugin_ids():
    config = cfg_plugin.plugin_config(pid)
    sources = cfg_plugin.plugin_config_sources(pid)
    print(f"{pid}:")
    for key, value in config.items():
        print(f"  {key}: {value!r}  [{sources[key]}]")
    print()

os.environ.pop("CE_PLUGIN_CONFIG_JSON", None)


Configured plugin IDs:

 ('demo.factual.v2', 'demo.interval.custom')

demo.factual.v2:
  threshold: 0.05  [env]
  max_features: 10  [env]

demo.interval.custom:
  confidence: 0.9  [env]
  method: 'venn_abers'  [env]



'{"demo.factual.v2": {"threshold": 0.05, "max_features": 10}, "demo.interval.custom": {"confidence": 0.9, "method": "venn_abers"}}'

In [10]:
# Simulating pyproject.toml plugin config via direct constructor
cfg_py_plugin = ConfigManager(
    env_snapshot={},
    pyproject_snapshot={
        "plugin_configs": {
            "demo.factual.v2": {"threshold": 0.05, "max_features": 10},
        }
    },
)

config = cfg_py_plugin.plugin_config("demo.factual.v2")
sources = cfg_py_plugin.plugin_config_sources("demo.factual.v2")
print("Plugin config from pyproject snapshot:")
for k, v in config.items():
    print(f"  {k}: {v!r}  [{sources[k]}]")


Plugin config from pyproject snapshot:
  threshold: 0.05  [pyproject]
  max_features: 10  [pyproject]


In [11]:
# Call-site overrides layer on top of the snapshot
cfg_ov2 = ConfigManager(
    env_snapshot={},
    pyproject_snapshot={
        "plugin_configs": {"demo.factual.v2": {"threshold": 0.05}}
    },
)

config_with_override = cfg_ov2.plugin_config(
    "demo.factual.v2",
    overrides={"threshold": 0.01, "extra_key": "runtime_value"},
)
sources_with_override = cfg_ov2.plugin_config_sources(
    "demo.factual.v2",
    overrides={"threshold": 0.01, "extra_key": "runtime_value"},
)
print("Config with call-site overrides (threshold overridden, extra_key added):")
for k, v in config_with_override.items():
    print(f"  {k}: {v!r}  [{sources_with_override[k]}]")


Config with call-site overrides (threshold overridden, extra_key added):
  threshold: 0.01  [override]
  extra_key: 'runtime_value'  [override]


## 6. Strict vs. permissive validation

By default `strict=True`: unknown pyproject keys and type mismatches raise
`ConfigurationError` immediately at construction.

`strict=False` collects all issues into a `ConfigValidationReport` so you can
surface every problem at once — useful during migration or staging validation.


In [12]:
# strict=True: bad pyproject config raises at construction
from calibrated_explanations.utils.exceptions import ConfigurationError

try:
    cfg_strict = ConfigManager(
        env_snapshot={},
        pyproject_snapshot={
            "plots": {"unknown_key": "value"},  # unknown key
        },
        strict=True,
    )
except ConfigurationError as exc:
    print(f"strict=True raised ConfigurationError:")
    print(f"  {exc}")


strict=True raised ConfigurationError:
  Unknown key(s) in [plots] configuration: ['unknown_key']


In [13]:
# strict=False: all issues captured without raising
with warnings.catch_warnings(record=True):
    warnings.simplefilter("always")
    cfg_lax = ConfigManager(
        env_snapshot={},
        pyproject_snapshot={
            "plots": {"unknown_key": "value"},  # unknown key in known section
            "explanations": {"factual": 42},   # type mismatch: must be str
        },
        strict=False,
    )

report = cfg_lax.validation_report()
print(f"Validation issues found: {len(report.issues)}")
print(f"has_errors:              {report.has_errors}")
print()
for issue in report.issues:
    print(f"  [{issue.location}] {issue.message}")


Validation issues found: 2
has_errors:              True

  [pyproject.plots] Unknown key(s) in [plots] configuration: ['unknown_key']
  [pyproject.explanations.factual] Invalid value for [explanations].factual: 42; must be a non-empty string


## 7. Process-level singleton lifecycle

`get_process_config_manager()` returns a lazily-constructed singleton shared
across the whole process.

`init_process_config_manager()` installs a custom manager before any runtime
component initializes it — call it once at a process boundary (e.g. server
startup). A second call raises `CalibratedError`.

`reset_process_config_manager_for_testing()` is **test-only** — it clears the
singleton so tests that mutate `os.environ` can start fresh. Production code
must never reset after initialization.


In [14]:
# Reset so this demo starts clean (test-only pattern)
reset_process_config_manager_for_testing()

# Lazy initialization: first call constructs and caches the singleton
singleton_a = get_process_config_manager()
print(f"type: {type(singleton_a).__name__}")

# Second call returns the exact same object
singleton_b = get_process_config_manager()
print(f"Same object: {singleton_a is singleton_b}")


type: ConfigManager
Same object: True


In [15]:
# init_process_config_manager(): install a custom manager at process startup
reset_process_config_manager_for_testing()

custom_manager = ConfigManager.from_sources(strict=False)
installed = init_process_config_manager(custom_manager)

print(f"Installed is the custom manager: {installed is custom_manager}")
print(f"get_process_config_manager() returns same: {get_process_config_manager() is custom_manager}")

# A second init call raises CalibratedError
from calibrated_explanations.utils.exceptions import CalibratedError

try:
    init_process_config_manager(custom_manager)
except CalibratedError as exc:
    print(f"Second init raises CalibratedError: {exc}")

# Restore a clean singleton for subsequent cells
reset_process_config_manager_for_testing()


Installed is the custom manager: True
get_process_config_manager() returns same: True
Second init raises CalibratedError: Process ConfigManager has already been initialized.


## 8. Telemetry diagnostic mode

`CE_TELEMETRY_DIAGNOSTIC_MODE` (env) or `[tool.calibrated_explanations.telemetry]
diagnostic_mode` (pyproject) enable structured telemetry payloads for audit
and governance workflows. Env takes precedence over pyproject per the
standard precedence order.


In [16]:
# Bool-like strings accepted in env
for raw_value in ("true", "1", "yes", "on", "false", "0", "no"):
    os.environ["CE_TELEMETRY_DIAGNOSTIC_MODE"] = raw_value
    cfg_tel = ConfigManager.from_sources()
    resolved = cfg_tel.telemetry_diagnostic_mode()
    print(f"  CE_TELEMETRY_DIAGNOSTIC_MODE={raw_value!r:6} -> {resolved}")

os.environ.pop("CE_TELEMETRY_DIAGNOSTIC_MODE", None)


  CE_TELEMETRY_DIAGNOSTIC_MODE='true' -> True


  CE_TELEMETRY_DIAGNOSTIC_MODE='1'    -> True
  CE_TELEMETRY_DIAGNOSTIC_MODE='yes'  -> True
  CE_TELEMETRY_DIAGNOSTIC_MODE='on'   -> True
  CE_TELEMETRY_DIAGNOSTIC_MODE='false' -> False


  CE_TELEMETRY_DIAGNOSTIC_MODE='0'    -> False
  CE_TELEMETRY_DIAGNOSTIC_MODE='no'   -> False


'no'

In [17]:
# pyproject sets it; env overrides it
cfg_py_tel = ConfigManager(
    env_snapshot={},
    pyproject_snapshot={"telemetry": {"diagnostic_mode": True}},
)
print(f"pyproject=True, no env   -> {cfg_py_tel.telemetry_diagnostic_mode()}")

cfg_env_beats = ConfigManager(
    env_snapshot={"CE_TELEMETRY_DIAGNOSTIC_MODE": "false"},
    pyproject_snapshot={"telemetry": {"diagnostic_mode": True}},
)
print(f"pyproject=True, env=false -> {cfg_env_beats.telemetry_diagnostic_mode()}  (env wins)")


pyproject=True, no env   -> True
pyproject=True, env=false -> False  (env wins)


## Summary

| Topic | Key API |
|-------|---------|
| Create from live env + pyproject | `ConfigManager.from_sources()` |
| Read one env value | `cfg.env("CE_PLOT_STYLE")` |
| Read a pyproject section | `cfg.pyproject_section("plots")` |
| Export diagnostic snapshot | `cfg.export_effective()` |
| Per-plugin config | `cfg.plugin_config(plugin_id)` |
| Per-key source attribution | `cfg.plugin_config_sources(plugin_id)` |
| Validation report | `cfg.validation_report()` |
| Telemetry mode | `cfg.telemetry_diagnostic_mode()` |
| Process singleton (lazy) | `get_process_config_manager()` |
| Process singleton (explicit) | `init_process_config_manager(manager)` |
| Reset for tests only | `reset_process_config_manager_for_testing()` |

**Key rules (ADR-034)**

- All runtime configuration reads go through `ConfigManager`.
- Never call `os.getenv` or parse `pyproject.toml` directly in non-boundary modules.
- `ConfigManager` is snapshot-based — reconstruct to pick up env changes.
- Construct once per runtime boundary; do not reconstruct on every lookup.
- `strict=True` (default) raises on bad config; `strict=False` collects issues.

**See also**

- `docs/foundations/how-to/configure_runtime.md` — full how-to guide
- `development/adrs/ADR-034-centralized-configuration-management.md` — ADR
